In [60]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
import numpy as np
import os, site
from datetime import datetime
from datetime import timedelta
import stonesoup

from stonesoup.models.transition.nonlinear import CTRV
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from stonesoup.models.measurement.nonlinear import CartesianToBearingRange
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import Detection

In [62]:
dt=1/15

In [63]:
start_time   = datetime.now().replace(microsecond=0)
current_time = start_time + timedelta(seconds=dt)
np.random.seed(1991)

transition_model = CTRV(linear_noise_coeff=0.5,turn_noise_coeff=0.1)
timesteps        = [current_time]
truth            = GroundTruthPath([GroundTruthState([0, 0, np.pi/4, 0.1, np.pi/16], timestamp=current_time)])

# %%
# Create the truth path
for k in range(1, 120):
    current_time += timedelta(seconds=dt)
    timesteps.append(current_time)
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=dt)),
        timestamp=current_time))

# %%
# Plot the ground truth.

from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truth, [0, 1])
plotter.fig

In [64]:
measurement_model = LinearGaussian(
    ndim_state=5,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 1),  # Mapping measurement vector index to state index
    noise_covar=np.array([[0.05, 0],  # Covariance matrix for Gaussian PDF
                          [0, 0.05]])
    )

# %%
# Populate the measurement array
measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement, timestamp=state.timestamp,measurement_model=measurement_model))

# %%
# Plot those measurements

plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_measurements(measurements, [0, 1])
plotter.plot_ground_truths(truth, [0, 1])
plotter.fig

In [65]:
from stonesoup.predictor.kalman import ExtendedKalmanPredictor
from stonesoup.updater.kalman import ExtendedKalmanUpdater

transition_model = CTRV(linear_noise_coeff=1,turn_noise_coeff=1)
predictor = ExtendedKalmanPredictor(transition_model)
updater = ExtendedKalmanUpdater(measurement_model)

from stonesoup.types.state import GaussianState
prior = GaussianState([[0], [0], [0], [0], [0]], np.diag([10, 10, 2*np.pi, 4,  2*np.pi]), timestamp=start_time)


In [66]:
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

track = Track()
for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis)
    track.append(post)
    prior = track[-1]

# %%
# Plot the resulting track with the sample points at each iteration. Can also change 'plot_history'
# to True if wanted.

plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_measurements(measurements, [0, 1])
plotter.plot_ground_truths(truth, [0, 1])
plotter.plot_tracks(track, [0, 1], plot_history=False,uncertainty=True)
plotter.fig



In [67]:
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truth, [0, 1])
plotter.plot_track_headings(track, [0, 1, 3, 2], plot_history=True, velocity_scale=0.1)
plotter.fig